In [1]:
import os
import time
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
from tqdm import tqdm
import pubchempy as pcp


RDLogger.DisableLog('rdApp.*') 

# ==========================================
# 1. CORE HELPER FUNCTIONS
# ==========================================

def get_canonical(smiles):
    if pd.isna(smiles) or not isinstance(smiles, str): return None
    try:
        mol = Chem.MolFromSmiles(smiles.strip())
        if mol: return Chem.MolToSmiles(mol, canonical=True)
    except: pass
    return None

def fetch_smiles_from_pubchem(identifier):
    if pd.isna(identifier) or str(identifier).strip() == "": return None
    try:
        compounds = pcp.get_compounds(str(identifier).strip(), 'name')
        if compounds: return compounds[0].isomeric_smiles
    except: pass
    return None

def clean_fda_identifier(identifier):
    """Strips FDA syntax weirdness so PubChem can read it."""
    ident = str(identifier).strip()
    ident = ident.replace('.ALPHA.-', 'Alpha-').replace('.BETA.-', 'Beta-').replace('.GAMMA.-', 'Gamma-')
    return ident.strip(' .')

def extract_smiles_from_file(filepath):
    if not os.path.exists(filepath): return set()
    print(f"   -> Loading pre-processed {filepath}...")
    try:
        df = pd.read_csv(filepath, on_bad_lines='skip', engine='python')
        if 'SMILES' not in df.columns: return set()
        raw_smiles = df['SMILES'].dropna().tolist()
        clean_set = set()
        for smi in tqdm(raw_smiles, desc=f"   Parsing {os.path.basename(filepath)}", leave=False):
            canonical = get_canonical(smi)
            if canonical: clean_set.add(canonical)
        return clean_set
    except: return set()

from concurrent.futures import ThreadPoolExecutor

def smart_bridge_to_pubchem(raw_file, processed_file, id_col, fallback_col, read_all_sheets=False, is_fda=False):
    if os.path.exists(processed_file):
        return extract_smiles_from_file(processed_file)
        
    print(f"   -> Hitting PubChem API (Multi-threaded mode) for {raw_file}...")
    try:
        # Load data exactly as before
        if raw_file.endswith('.csv'):
            df = pd.read_csv(raw_file, on_bad_lines='skip', engine='python')
        else:
            if read_all_sheets:
                all_sheets = pd.read_excel(raw_file, sheet_name=None)
                df = pd.concat(all_sheets.values(), ignore_index=True)
            else:
                df = pd.read_excel(raw_file)
        
        df.columns = df.columns.str.strip()
        actual_id_col = next((c for c in df.columns if id_col.lower() in c.lower()), None)
        actual_fb_col = next((c for c in df.columns if fallback_col.lower() in c.lower()), None)
        
        unique_rows = df[[actual_id_col, actual_fb_col]].drop_duplicates()
        smiles_dict = {}

        # Worker function for threading
        def fetch_worker(row):
            cas = str(row[actual_id_col]).strip()
            name = str(row[actual_fb_col]).strip()
            target = clean_fda_identifier(name) if (is_fda and (cas == 'nan' or '-' not in cas)) else cas
            if target == 'nan': target = name
            
            result = fetch_smiles_from_pubchem(target)
            time.sleep(0.2) # Keeping this to stay under the 5 req/sec limit
            return (target, result)

        # Threaded Execution (5 concurrent workers)
        with ThreadPoolExecutor(max_workers=5) as executor:
            results = list(tqdm(executor.map(fetch_worker, [row for _, row in unique_rows.iterrows()]), 
                                total=len(unique_rows), desc=f"   PubChem API (Threading)"))
        
        smiles_dict = dict(results)
        
        # Map back
        def map_row(row):
            c = str(row[actual_id_col]).strip()
            target = clean_fda_identifier(row[actual_fb_col]) if (is_fda and (c == 'nan' or '-' not in c)) else c
            if target == 'nan': target = row[actual_fb_col]
            return smiles_dict.get(target)
            
        df['SMILES'] = df.apply(map_row, axis=1)
        df.to_csv(processed_file, index=False)
        return extract_smiles_from_file(processed_file)
    except Exception as e:
        print(f"   ❌ Error building Bridge for {raw_file}: {e}")
        return set()
# ==========================================
# 2. MAIN EXECUTION PIPELINE
# ==========================================

if __name__ == "__main__":
    master_safe_set = set()

    print("\n==================================================")
    print(" PHASE 1: BUILDING THE WHITELIST (SAFE SHIELD)")
    print("==================================================")

    # 1. EPA SCIL 
    print("\n1. Processing EPA SCIL (Industrial Fillers)...")
    master_safe_set.update(smart_bridge_to_pubchem(
        "safer_chemical_ingredients_list.xls", "epa_scil_with_smiles.csv", 
        id_col="CAS", fallback_col="Chemical Name", read_all_sheets=True
    ))

    # 2. FDA SCOGS 
    print("\n2. Processing FDA SCOGS (Food Safe)...")
    master_safe_set.update(extract_smiles_from_file("scogs_with_smiles.csv"))

    # 3. FDA IIR 
    print("\n3. Processing FDA IIR (Inactive Pharma Ingredients)...")
    master_safe_set.update(smart_bridge_to_pubchem(
        "IIR_OCOMM.csv", "fda_iir_with_smiles.csv", 
        id_col="CAS_NUMBER", fallback_col="INGREDIENT", is_fda=True
    ))

    # 4. FooDB
    print("\n4. Processing FooDB (Local structures.sdf)...")
    if os.path.exists("structures.sdf"):
        try:
            supplier = Chem.SDMolSupplier("structures.sdf")
            for mol in tqdm(supplier, desc="   Parsing SDF", leave=False):
                if mol is not None:
                    master_safe_set.add(Chem.MolToSmiles(mol, canonical=True))
        except Exception as e:
             print(f"   ❌ Error parsing SDF: {e}")
    else:
        print("   ⚠️ Missing: structures.sdf")

    # 5. ChEBI (Restored to catch rogue vitamins and amino acids)
    print("\n5. Processing ChEBI (Endogenous Biology)...")
    if all(os.path.exists(f) for f in ["structures.tsv.gz", "relation.tsv.gz"]):
        try:
            print("   -> Loading ChEBI Relations...")
            rel_df = pd.read_csv("relation.tsv.gz", sep="\t", compression="gzip", on_bad_lines='skip', engine='python')
            
            # Safe roles: Amino acids (36080), primary metabolites (48706), carbohydrates (25367), vitamins (27303)
            safe_roles = [36080, 48706, 25367, 27303] 
            safe_ids = set(rel_df[rel_df['final_id'].isin(safe_roles)]['init_id'])
            
            print("   -> Loading ChEBI Structures...")
            struct_df = pd.read_csv("structures.tsv.gz", sep="\t", compression="gzip", on_bad_lines='skip', engine='python')
            chebi_safe = struct_df[struct_df['compound_id'].isin(safe_ids)]
            
            if 'smiles' in chebi_safe.columns:
                for smi in tqdm(chebi_safe['smiles'].dropna(), desc="   Parsing ChEBI", leave=False):
                    canonical = get_canonical(smi)
                    if canonical: master_safe_set.add(canonical)
            else:
                print("   ❌ 'smiles' column missing from structures.tsv.gz")
                
        except Exception as e:
            print(f"   ❌ Error processing ChEBI: {e}")
    else:
         print("   ⚠️ Missing ChEBI .tsv.gz files. Skipping.")

    print(f"\n=> Total Safe Molecules Accumulated: {len(master_safe_set)}")

    print("\n==================================================")
    print(" PHASE 2: APPLYING THE BLACKLIST FILTER")
    print("==================================================")
    
    print("\nExtracting known toxins...")
    if os.path.exists("master_toxocity_dataset.csv"):
        tox_df = pd.read_csv("master_toxocity_dataset.csv", on_bad_lines='skip', engine='python')
        target_col = next((col for col in tox_df.columns if 'SMILES' in str(col).upper()), None)
        
        tox_set = set()
        if target_col:
            for smi in tqdm(tox_df[target_col].dropna(), desc="   Parsing Toxins", leave=False):
                canonical = get_canonical(smi)
                if canonical: tox_set.add(canonical)
                
        print(f"=> Total Known Toxins: {len(tox_set)}")

        overlap = master_safe_set.intersection(tox_set)
        final_clean_smiles = master_safe_set - tox_set

        print("\n==================================================")
        print(" FINAL REPORT")
        print("==================================================")
        print(f"Safe Molecules Found:   {len(master_safe_set)}")
        print(f"Toxins Intercepted:     {len(overlap)}")
        print(f"FINAL SHIELD SIZE:      {len(final_clean_smiles)}")

        pd.DataFrame({'SMILES': list(final_clean_smiles)}).to_csv("ultimate_safety_shield.csv", index=False)
        print("\n✅ Saved successfully to 'ultimate_safety_shield.csv'.")
    else:
        print("❌ 'master_toxocity_dataset.csv' missing. Cannot apply filter.")


 PHASE 1: BUILDING THE WHITELIST (SAFE SHIELD)

1. Processing EPA SCIL (Industrial Fillers)...
   -> Hitting PubChem API (Multi-threaded mode) for safer_chemical_ingredients_list.xls...


   PubChem API (Threading):   0%|          | 0/1172 [00:00<?, ?it/s]/var/folders/sp/ggmsl14s3r11zy9plmylcts00000gn/T/ipykernel_61505/2198029958.py:28: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  if compounds: return compounds[0].isomeric_smiles
   PubChem API (Threading): 100%|██████████| 1172/1172 [05:29<00:00,  3.56it/s]


   -> Loading pre-processed epa_scil_with_smiles.csv...



2. Processing FDA SCOGS (Food Safe)...
   -> Loading pre-processed scogs_with_smiles.csv...



3. Processing FDA IIR (Inactive Pharma Ingredients)...
   -> Hitting PubChem API (Multi-threaded mode) for IIR_OCOMM.csv...


...

   -> Loading pre-processed fda_iir_with_smiles.csv...



4. Processing FooDB (Local structures.sdf)...



5. Processing ChEBI (Endogenous Biology)...
   -> Loading ChEBI Relations...
   -> Loading ChEBI Structures...



=> Total Safe Molecules Accumulated: 218440

 PHASE 2: APPLYING THE BLACKLIST FILTER

Extracting known toxins...


=> Total Known Toxins: 37467

 FINAL REPORT
Safe Molecules Found:   218440
Toxins Intercepted:     6639
FINAL SHIELD SIZE:      211801

✅ Saved successfully to 'ultimate_safety_shield.csv'.
